
# LA Building Permits Analytics Workflow

This notebook demonstrates an end-to-end analytics workflow for
analyzing building permit data, from data preparation and exploration
to insight generation.
- Permit records:&nbsp;
https://data.lacity.org/A-Prosperous-City/Building-Permits-Since-2012/vdg9-hy7c/about_data
- Census tracts:&nbsp;
https://data.lacounty.gov/datasets/lacounty::median-income-and-ami-census-tract/about
- Contractors:&nbsp;
https://www.cslb.ca.gov/onlineservices/dataportal/ContractorList


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
session.sql("USE ROLE TRAINING_ROLE").collect() 
session.sql("USE DATABASE LA_PERMIT_DATA").collect()
session.sql("USE SCHEMA PUBLIC").collect()

In [ ]:
records        = session.table("PERMIT_RECORDS").to_pandas()
census_tracts  = session.table("CENSUS_TRACTS").to_pandas()
contractors    = session.table("MASTER_LICENSE").to_pandas()
personnel      = session.table("PERSONNEL_DATA").to_pandas()
worker_comp    = session.table("WORKER_COMP").to_pandas()

In [ ]:
# 3.1 Dates
for col in ["ISSUE_DATE", "STATUS_DATE"]:
    records[col] = pd.to_datetime(records[col], errors="coerce")

# 3.2 Valuation (string -> numeric)
records["VALUATION"] = pd.to_numeric(
        records["VALUATION"].str.replace("$", "").str.replace(",",""),
        errors="coerce"
    )

# 3.3 How ot get a specific permit type, for example here we get HVAC (Heating, ventilation, and air conditioning)

hvac = records.loc[records["PERMIT_TYPE"].fillna("").str.upper() == "HVAC"]

# 3.5 Derive CT (LA‑specific tract key) from CENSUS_TRACT if present
ct_numeric = pd.to_numeric(records["CENSUS_TRACT"], errors="coerce")
records["CT"] = (ct_numeric * 100) + 6037000000

In [ ]:

records_with_census = records.merge(census_tracts, left_on="CT", right_on="CENSUS_TRACT", how="inner")

In [ ]:
hvac_with_contractors = pd.DataFrame()


hvac_by_contractor = (
            hvac.groupby("LICENSE_NUM")
                .agg(Projects=("PERMIT_SUB_TYPE", "count"),
                     TotalValue=("VALUATION", "sum"),
                     FirstYear=("ISSUE_DATE", "min"),
                     LastYear=("ISSUE_DATE", "max"))
                .reset_index()
        )

hvac_with_contractors = hvac_by_contractor.merge(
            contractors, left_on="LICENSE_NUM", right_on="LICENSE_NO", how="left"
        )


In [ ]:

pool_permits = records[records["AI_DESCRIPTION"].fillna("").str.contains("swimming pool", case=False)].copy()
pool_permits["year"] = pool_permits["ISSUE_DATE"].dt.year

counts_by_year = pool_permits.groupby("year")["AI_DESCRIPTION"].count().rename("num_permits")
value_by_year  = pool_permits.groupby("year")["VALUATION"].sum().rename("total_valuation") 




plt.figure()
counts_by_year.sort_index().plot(kind="bar")
plt.title("Swimming Pool Permits per Year")
plt.xlabel("Year")
plt.ylabel("Number of Permits")
plt.tight_layout()
plt.show()


In [ ]:
pool_by_zip = (pool_permits.groupby("ZIP_CODE")["AI_DESCRIPTION"]
                   .count()
                   .rename("num_pool_permits")
                   .sort_values(ascending=False)
                   .head(15))



plt.figure()
pool_by_zip.sort_values(ascending=True).plot(kind="barh")
plt.title("Top ZIP Codes by Pool Permits")
plt.xlabel("Number of Permits")
plt.ylabel("ZIP Code")
plt.tight_layout()
plt.show()


In [ ]:
group = "AMI_CATEGORY"
kitchen = records_with_census[
        records_with_census["AI_DESCRIPTION"].fillna("").str.contains("kitchen")
    ].copy()

kitchen_by_ami = kitchen[group].value_counts().rename("permits").sort_index()
hh_by_ami = (records_with_census.groupby(group)["NUM_HOUSEHOLDS"]
                     .sum()
                     .rename("households")
                     .sort_index())

dist = pd.concat([kitchen_by_ami / kitchen_by_ami.sum(),
                          hh_by_ami / hh_by_ami.sum()], axis=1)
dist.columns = ["permit_share", "household_share"]


# Compare via simple side-by-side bars (two panels)
ax = dist.plot(kind="bar")
ax.set_title("Kitchen Permits vs Household Share by AMI Category")
ax.set_xlabel("AMI Category")
ax.set_ylabel("Share")
plt.tight_layout()
plt.show()


In [ ]:


# Pool filter
pool = records[records["AI_DESCRIPTION"].fillna("").str.contains("swimming pool")].copy()

# Sidebar filters
st.sidebar.header("Filters")
min_dt = pd.to_datetime(pool["ISSUE_DATE"]).min().date()
max_dt = pd.to_datetime(pool["ISSUE_DATE"]).max().date()
date_range = st.sidebar.date_input("Issue date range", value=(min_dt, max_dt), min_value=min_dt, max_value=max_dt)


zip_options = sorted(pool["ZIP_CODE"].dropna().unique().tolist()) if "ZIP_CODE" in pool.columns else []
selected_zips = st.sidebar.multiselect("ZIP codes", options=zip_options, default=zip_options[:5] if zip_options else [])

# Apply filters
mask = pd.Series(True, index=pool.index)
if date_range:
    start, end = date_range
    mask &= pool["ISSUE_DATE"].dt.date.between(start, end)

if selected_zips:
    mask &= pool["ZIP_CODE"].isin(selected_zips)

filtered = pool.loc[mask].copy()

st.subheader("Filtered Pool Permits")
st.write(f"{len(filtered):,} permits match your filters.")

cols = ["PCIS_PERMIT_NUM","ISSUE_DATE","STATUS","VALUATION","AI_DESCRIPTION","ADDRESS_START","STREET_NAME","ZIP_CODE"] 
st.dataframe(filtered[cols].sort_values("ISSUE_DATE", ascending=False), use_container_width=True)


# Coordinates for map (LATITUDE_LONGITUDE like '(lon lat)')

s=filtered['LATITUDE_LONGITUDE'].dropna()
coords=s.str.extract(r'\(([-\d.]+) ([-\d.]+)\)').rename(columns={0:'longitude',1:'latitude'}).astype(float)

st.subheader("Map")
st.map(coords)